# Beat Timer v0.3

**Plug-and-play workflow:** set `AUDIO_PATH` below, then run all cells.

The notebook will:
1. load the audio,
2. compute spectral flux,
3. estimate BPM and phase,
4. match onset evidence to the beat grid,
5. find the minimum number of timing sections,
6. render a scrollable segment visualizer.


In [3]:
from pathlib import Path

import librosa
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots


## Configuration

In [ ]:
# Change only this path for a new song.
AUDIO_PATH = Path("sakuzyo_that_day.mp3")

# Analysis quality. Keep 256 while developing; use 32 or 16 for a final precision pass.
N_FFT = 2048
HOP_LENGTH = 256

# BPM search range.
MIN_BPM = 80
MAX_BPM = 240
ROUND_INITIAL_BPM = 1

# Rising-onset detector.
MINIMUM_RISE = 0.20
MINIMUM_SLOPE = 0.02
LOOKBACK_FRAMES = 30
MINIMUM_PEAK_DISTANCE_FRAMES = 30

# Grid phase and onset matching.
PHASE_STEPS = 2000
PHASE_MATCH_TOLERANCE_SECONDS = 0.040
GRID_MATCH_TOLERANCE_SECONDS = 0.060

# Timing-section model.
USE_INTEGER_BPM = True
SECTION_TOLERANCE_MS = 6.0
MINIMUM_SECTION_POINTS = 8
ERROR_QUANTILE = 0.95

# Visualizer width. A 3-minute song at 45 px/s becomes 8100 px wide and scrollable.
PIXELS_PER_SECOND = 45
MINIMUM_FIGURE_WIDTH = 1200


## Core functions

In [5]:
def parabolic_peak_offset(values: np.ndarray, index: int) -> float:
    if index <= 0 or index >= len(values) - 1:
        return 0.0

    left = float(values[index - 1])
    center = float(values[index])
    right = float(values[index + 1])
    denominator = left - 2 * center + right

    if abs(denominator) < 1e-12:
        return 0.0

    return 0.5 * (left - right) / denominator


def detect_rising_onsets(
    data: np.ndarray,
    minimum_rise: float = 0.20,
    minimum_slope: float = 0.02,
    lookback: int = 30,
    minimum_distance: int = 20,
) -> tuple[list[int], dict[int, dict[str, float | int]]]:
    x = np.asarray(data, dtype=float)
    candidates: list[int] = []
    features: dict[int, dict[str, float | int]] = {}

    for peak in range(1, len(x) - 1):
        is_local_peak = x[peak] >= x[peak - 1] and x[peak] > x[peak + 1]
        if not is_local_peak:
            continue

        start = max(0, peak - lookback)
        if start >= peak:
            continue

        valley = start + int(np.argmin(x[start:peak]))
        rise = float(x[peak] - x[valley])
        rise_time = peak - valley
        slope = rise / max(rise_time, 1)

        if rise >= minimum_rise and slope >= minimum_slope:
            candidates.append(peak)
            features[peak] = {
                "valley": valley,
                "rise": rise,
                "rise_time": rise_time,
                "slope": float(slope),
            }

    selected: list[int] = []
    ranked = sorted(
        candidates,
        key=lambda p: (features[p]["rise"], features[p]["slope"]),
        reverse=True,
    )

    for peak in ranked:
        if all(abs(peak - chosen) >= minimum_distance for chosen in selected):
            selected.append(peak)

    return sorted(selected), features


def estimate_bpm_from_flux(
    spectral_flux: np.ndarray,
    sample_rate: int,
    hop_length: int,
    min_bpm: float,
    max_bpm: float,
) -> dict:
    if min_bpm <= 0 or max_bpm <= min_bpm:
        raise ValueError("Require 0 < min_bpm < max_bpm.")

    onset_signal = spectral_flux - np.mean(spectral_flux)
    autocorrelation = np.correlate(onset_signal, onset_signal, mode="full")
    autocorrelation = autocorrelation[len(autocorrelation) // 2:]

    frames_per_second = sample_rate / hop_length
    minimum_lag = max(1, int(np.floor(frames_per_second * 60 / max_bpm)))
    maximum_lag = min(
        len(autocorrelation) - 2,
        int(np.ceil(frames_per_second * 60 / min_bpm)),
    )

    if maximum_lag <= minimum_lag:
        raise ValueError("Audio is too short for the requested BPM range.")

    lags = np.arange(minimum_lag, maximum_lag + 1)
    search_region = autocorrelation[minimum_lag:maximum_lag + 1]
    best_local_index = int(np.argmax(search_region))
    best_integer_lag = int(lags[best_local_index])
    fractional_offset = parabolic_peak_offset(search_region, best_local_index)
    best_fractional_lag = best_integer_lag + fractional_offset

    period = best_fractional_lag / frames_per_second
    bpm = 60 / period

    return {
        "bpm": float(bpm),
        "period": float(period),
        "integer_lag": best_integer_lag,
        "fractional_lag": float(best_fractional_lag),
        "autocorrelation": autocorrelation,
        "lags": lags,
        "search_region": search_region,
    }


def estimate_grid_phase(
    peak_times: np.ndarray,
    peak_strengths: np.ndarray,
    period: float,
    phase_steps: int = 2000,
    tolerance: float = 0.040,
) -> dict:
    phases = np.linspace(0, period, phase_steps, endpoint=False)
    phase_scores = np.zeros_like(phases)

    for i, phase in enumerate(phases):
        distances = np.abs(((peak_times - phase + period / 2) % period) - period / 2)
        matches = distances <= tolerance

        if np.any(matches):
            timing_weight = 1 - distances[matches] / tolerance
            phase_scores[i] = np.sum(peak_strengths[matches] * timing_weight)

    best_index = int(np.argmax(phase_scores))
    return {
        "phase": float(phases[best_index]),
        "score": float(phase_scores[best_index]),
        "phases": phases,
        "phase_scores": phase_scores,
    }


def match_onsets_to_grid(
    peak_times: np.ndarray,
    predicted_beats: np.ndarray,
    tolerance: float,
) -> dict:
    matched_beat_indices: list[int] = []
    matched_grid_times: list[float] = []
    matched_peak_times: list[float] = []
    timing_errors: list[float] = []

    if len(peak_times) == 0:
        raise ValueError("No onset peaks were detected.")

    for beat_index, grid_time in enumerate(predicted_beats):
        nearest_index = int(np.argmin(np.abs(peak_times - grid_time)))
        nearest_peak = float(peak_times[nearest_index])
        error = nearest_peak - float(grid_time)

        if abs(error) <= tolerance:
            matched_beat_indices.append(beat_index)
            matched_grid_times.append(float(grid_time))
            matched_peak_times.append(nearest_peak)
            timing_errors.append(error)

    return {
        "beat_indices": np.asarray(matched_beat_indices, dtype=int),
        "grid_times": np.asarray(matched_grid_times, dtype=float),
        "peak_times": np.asarray(matched_peak_times, dtype=float),
        "errors": np.asarray(timing_errors, dtype=float),
    }


def fit_linear_grid(
    beat_indices: np.ndarray,
    beat_times: np.ndarray,
    start: int,
    end: int,
    integer_bpm: bool = False,
) -> tuple[float, float, np.ndarray]:
    indices = beat_indices[start:end + 1].astype(float)
    times = beat_times[start:end + 1].astype(float)

    if len(indices) < 2 or np.ptp(indices) == 0:
        raise ValueError("At least two distinct beat indices are required.")

    design = np.column_stack([np.ones_like(indices), indices])
    offset, period = np.linalg.lstsq(design, times, rcond=None)[0]

    if not np.isfinite(period) or period <= 0:
        raise ValueError("The fitted period is invalid.")

    if integer_bpm:
        fitted_bpm = 60 / period
        if not np.isfinite(fitted_bpm) or fitted_bpm <= 0:
            raise ValueError("The fitted BPM is invalid.")

        rounded_bpm = int(round(fitted_bpm))
        if rounded_bpm <= 0:
            raise ValueError("Rounded BPM must be positive.")

        period = 60 / rounded_bpm
        offset = float(np.mean(times - period * indices))

    predicted = offset + period * indices
    residuals = times - predicted
    return float(offset), float(period), residuals


def minimum_timing_sections(
    beat_indices: np.ndarray,
    beat_times: np.ndarray,
    tolerance_seconds: float = 0.010,
    minimum_section_points: int = 4,
    error_quantile: float = 0.95,
    integer_bpm: bool = False,
) -> list[dict]:
    beat_indices = np.asarray(beat_indices, dtype=int)
    beat_times = np.asarray(beat_times, dtype=float)

    if beat_indices.shape != beat_times.shape:
        raise ValueError("beat_indices and beat_times must have the same shape.")
    if not 0 < error_quantile <= 1:
        raise ValueError("error_quantile must be between 0 and 1.")
    if minimum_section_points < 2:
        raise ValueError("minimum_section_points must be at least 2.")

    n = len(beat_times)
    if n == 0:
        return []
    if n < minimum_section_points:
        raise ValueError("Not enough matched beats for one timing section.")

    infinity = n + 1
    best_count = np.full(n + 1, infinity, dtype=int)
    best_error = np.full(n + 1, np.inf, dtype=float)
    previous = np.full(n + 1, -1, dtype=int)
    best_count[0] = 0
    best_error[0] = 0.0
    segment_cache: dict[tuple[int, int], dict[str, float]] = {}

    for end_exclusive in range(1, n + 1):
        for start in range(end_exclusive):
            length = end_exclusive - start
            if length < minimum_section_points:
                continue
            if best_count[start] == infinity:
                continue

            try:
                offset, period, residuals = fit_linear_grid(
                    beat_indices,
                    beat_times,
                    start,
                    end_exclusive - 1,
                    integer_bpm=integer_bpm,
                )
            except ValueError:
                continue

            if not np.all(np.isfinite(residuals)):
                continue

            absolute_errors = np.abs(residuals)
            representative_error = float(np.quantile(absolute_errors, error_quantile))
            max_error = float(np.max(absolute_errors))

            if not np.isfinite(representative_error) or not np.isfinite(max_error):
                continue

            segment_cache[(start, end_exclusive)] = {
                "offset": offset,
                "period": period,
                "representative_error": representative_error,
                "max_error": max_error,
            }

            if representative_error > tolerance_seconds:
                continue

            candidate_count = best_count[start] + 1
            segment_error = float(np.sum(residuals ** 2))
            candidate_error = best_error[start] + segment_error

            better_count = candidate_count < best_count[end_exclusive]
            same_count_better_fit = (
                candidate_count == best_count[end_exclusive]
                and candidate_error < best_error[end_exclusive]
            )

            if better_count or same_count_better_fit:
                best_count[end_exclusive] = candidate_count
                best_error[end_exclusive] = candidate_error
                previous[end_exclusive] = start

    if previous[n] == -1:
        raise ValueError(
            "No valid segmentation found. Increase SECTION_TOLERANCE_MS, "
            "reduce MINIMUM_SECTION_POINTS, or improve onset matching."
        )

    sections: list[dict] = []
    end_exclusive = n

    while end_exclusive > 0:
        start = int(previous[end_exclusive])
        if start < 0:
            raise RuntimeError("Segmentation backtracking failed.")

        cached = segment_cache[(start, end_exclusive)]
        offset = cached["offset"]
        period = cached["period"]
        representative_error = cached["representative_error"]
        max_error = cached["max_error"]

        if representative_error > tolerance_seconds + 1e-12:
            raise RuntimeError("Internal error: returned section exceeds tolerance.")

        bpm = 60 / period
        bpm_value = int(round(bpm)) if integer_bpm else float(bpm)

        sections.append({
            "start_observation": start,
            "end_observation": end_exclusive - 1,
            "start_beat_index": int(beat_indices[start]),
            "end_beat_index": int(beat_indices[end_exclusive - 1]),
            "start_time": float(beat_times[start]),
            "end_time": float(beat_times[end_exclusive - 1]),
            "offset": float(offset),
            "period": float(period),
            "bpm": bpm_value,
            "quantile_error_ms": representative_error * 1000,
            "max_error_ms": max_error * 1000,
            "number_of_points": end_exclusive - start,
        })

        end_exclusive = start

    return list(reversed(sections))


## One-call analysis pipeline

In [6]:
def analyze_song(
    audio_path: Path | str,
    *,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
    min_bpm: float = MIN_BPM,
    max_bpm: float = MAX_BPM,
    round_initial_bpm: bool = ROUND_INITIAL_BPM,
    integer_section_bpm: bool = USE_INTEGER_BPM,
    section_tolerance_ms: float = SECTION_TOLERANCE_MS,
) -> dict:
    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path.resolve()}")

    audio, sample_rate = librosa.load(audio_path, sr=None, mono=True)
    duration = len(audio) / sample_rate

    stft = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    magnitude = np.abs(stft)
    positive_difference = np.maximum(magnitude[:, 1:] - magnitude[:, :-1], 0)
    spectral_flux = np.concatenate([[0.0], np.sum(positive_difference, axis=0)])
    spectral_flux /= np.max(spectral_flux) + 1e-12

    flux_times = librosa.frames_to_time(
        np.arange(len(spectral_flux)),
        sr=sample_rate,
        hop_length=hop_length,
    )

    rise_peaks, rise_features = detect_rising_onsets(
        spectral_flux,
        minimum_rise=MINIMUM_RISE,
        minimum_slope=MINIMUM_SLOPE,
        lookback=LOOKBACK_FRAMES,
        minimum_distance=MINIMUM_PEAK_DISTANCE_FRAMES,
    )

    if not rise_peaks:
        raise ValueError("No rising onsets were detected. Relax the onset settings.")

    peak_times = flux_times[rise_peaks]
    peak_strengths = spectral_flux[rise_peaks]

    bpm_result = estimate_bpm_from_flux(
        spectral_flux,
        sample_rate,
        hop_length,
        min_bpm,
        max_bpm,
    )

    initial_bpm = round(bpm_result["bpm"]) if round_initial_bpm else bpm_result["bpm"]
    period = 60 / initial_bpm

    phase_result = estimate_grid_phase(
        peak_times,
        peak_strengths,
        period,
        phase_steps=PHASE_STEPS,
        tolerance=PHASE_MATCH_TOLERANCE_SECONDS,
    )

    predicted_beats = np.arange(phase_result["phase"], duration + period, period)
    matches = match_onsets_to_grid(
        peak_times,
        predicted_beats,
        tolerance=GRID_MATCH_TOLERANCE_SECONDS,
    )

    sections = minimum_timing_sections(
        beat_indices=matches["beat_indices"],
        beat_times=matches["peak_times"],
        tolerance_seconds=section_tolerance_ms / 1000,
        minimum_section_points=MINIMUM_SECTION_POINTS,
        error_quantile=ERROR_QUANTILE,
        integer_bpm=integer_section_bpm,
    )

    return {
        "audio_path": audio_path,
        "audio": audio,
        "sample_rate": sample_rate,
        "duration": duration,
        "n_fft": n_fft,
        "hop_length": hop_length,
        "spectral_flux": spectral_flux,
        "flux_times": flux_times,
        "rise_peaks": np.asarray(rise_peaks, dtype=int),
        "rise_features": rise_features,
        "peak_times": peak_times,
        "peak_strengths": peak_strengths,
        "bpm_result": bpm_result,
        "initial_bpm": initial_bpm,
        "initial_period": period,
        "phase_result": phase_result,
        "predicted_beats": predicted_beats,
        "matches": matches,
        "sections": sections,
        "section_tolerance_ms": section_tolerance_ms,
    }


def print_analysis_summary(result: dict) -> None:
    matches = result["matches"]
    errors_ms = np.abs(matches["errors"]) * 1000

    print(f"File: {result['audio_path']}")
    print(f"Duration: {result['duration']:.3f} s")
    print(f"Sample rate: {result['sample_rate']} Hz")
    print(f"Hop length: {result['hop_length']} samples")
    print(f"Autocorrelation BPM: {result['bpm_result']['bpm']:.6f}")
    print(f"Initial grid BPM: {result['initial_bpm']}")
    print(f"Initial phase: {result['phase_result']['phase']:.6f} s")
    print(f"Detected onset candidates: {len(result['peak_times'])}")
    print(f"Matched beat observations: {len(matches['peak_times'])}")

    if len(errors_ms):
        print(f"Initial-grid median error: {np.median(errors_ms):.3f} ms")
        print(f"Initial-grid 95% error: {np.percentile(errors_ms, 95):.3f} ms")
        print(f"Initial-grid maximum error: {np.max(errors_ms):.3f} ms")

    print(f"Timing sections: {len(result['sections'])}")
    for i, section in enumerate(result["sections"], start=1):
        print(
            f"  {i}: beats {section['start_beat_index']}–{section['end_beat_index']}, "
            f"BPM={section['bpm']}, offset={section['offset']:.6f}s, "
            f"95%={section['quantile_error_ms']:.3f}ms, "
            f"max={section['max_error_ms']:.3f}ms"
        )


## Scrollable segment visualizer

In [7]:
def _vertical_line_trace(
    x_positions: np.ndarray,
    y_min: float,
    y_max: float,
    *,
    name: str,
    width: float = 1.0,
    dash: str = "dot",
    showlegend: bool = True,
) -> go.Scatter:
    x: list[float | None] = []
    y: list[float | None] = []

    for value in x_positions:
        x.extend([float(value), float(value), None])
        y.extend([y_min, y_max, None])

    return go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name=name,
        line={"width": width, "dash": dash},
        showlegend=showlegend,
        hoverinfo="skip",
    )


def visualize_timing_segments(
    result: dict,
    *,
    pixels_per_second: int = PIXELS_PER_SECOND,
    minimum_width: int = MINIMUM_FIGURE_WIDTH,
    time_range: tuple[float, float] | None = None,
) -> go.Figure:
    duration = float(result["duration"])
    start_time, end_time = (0.0, duration) if time_range is None else time_range

    if start_time < 0 or end_time <= start_time:
        raise ValueError("time_range must satisfy 0 <= start < end.")

    width = max(minimum_width, int((end_time - start_time) * pixels_per_second))
    flux_times = result["flux_times"]
    spectral_flux = result["spectral_flux"]
    peak_times = result["peak_times"]
    peak_strengths = result["peak_strengths"]
    matches = result["matches"]
    sections = result["sections"]

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.62, 0.38],
        subplot_titles=("Onset evidence and timing sections", "Section residuals"),
    )

    flux_mask = (flux_times >= start_time) & (flux_times <= end_time)
    peak_mask = (peak_times >= start_time) & (peak_times <= end_time)

    fig.add_trace(
        go.Scatter(
            x=flux_times[flux_mask],
            y=spectral_flux[flux_mask],
            mode="lines",
            name="Spectral flux",
            line={"width": 1.2},
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=peak_times[peak_mask],
            y=peak_strengths[peak_mask],
            mode="markers",
            name="Onset candidates",
            marker={"symbol": "x", "size": 7},
        ),
        row=1,
        col=1,
    )

    for section_number, section in enumerate(sections, start=1):
        start_obs = section["start_observation"]
        end_obs = section["end_observation"]
        section_indices = matches["beat_indices"][start_obs:end_obs + 1]
        section_times = matches["peak_times"][start_obs:end_obs + 1]
        predicted_times = section["offset"] + section["period"] * section_indices
        residuals_ms = (section_times - predicted_times) * 1000

        visible = (section_times >= start_time) & (section_times <= end_time)
        if np.any(visible):
            fig.add_trace(
                go.Scatter(
                    x=section_times[visible],
                    y=residuals_ms[visible],
                    mode="markers",
                    name=f"Section {section_number}: {section['bpm']} BPM",
                    marker={"size": 7},
                    customdata=np.column_stack([
                        section_indices[visible],
                        predicted_times[visible],
                    ]),
                    hovertemplate=(
                        "time=%{x:.4f}s<br>residual=%{y:.3f}ms"
                        "<br>beat=%{customdata[0]}"
                        "<br>grid=%{customdata[1]:.4f}s<extra></extra>"
                    ),
                ),
                row=2,
                col=1,
            )

        first_index = section["start_beat_index"]
        last_index = section["end_beat_index"]
        section_grid = section["offset"] + section["period"] * np.arange(
            first_index,
            last_index + 1,
        )
        section_grid = section_grid[(section_grid >= start_time) & (section_grid <= end_time)]

        if len(section_grid):
            fig.add_trace(
                _vertical_line_trace(
                    section_grid,
                    0.0,
                    1.0,
                    name=f"Grid {section_number}",
                    width=0.7,
                    dash="dot",
                    showlegend=False,
                ),
                row=1,
                col=1,
            )

        boundary_time = section["offset"] + section["period"] * first_index
        if section_number > 1 and start_time <= boundary_time <= end_time:
            fig.add_vline(
                x=boundary_time,
                line_width=2,
                line_dash="dash",
                annotation_text=f"Section {section_number}: {section['bpm']} BPM",
                annotation_position="top",
                row="all",
                col=1,
            )

    tolerance = result["section_tolerance_ms"]
    fig.add_hline(y=0, line_dash="dot", row=2, col=1)
    fig.add_hline(y=tolerance, line_dash="dot", row=2, col=1)
    fig.add_hline(y=-tolerance, line_dash="dot", row=2, col=1)

    fig.update_xaxes(range=[start_time, end_time], title_text="Time (seconds)", row=2, col=1)
    fig.update_yaxes(title_text="Normalized flux", row=1, col=1)
    fig.update_yaxes(title_text="Residual (ms)", row=2, col=1)
    fig.update_layout(
        title=(
            f"{result['audio_path'].name} | "
            f"{len(sections)} timing section(s)"
        ),
        width=width,
        height=720,
        autosize=False,
        hovermode="closest",
        margin={"l": 70, "r": 30, "t": 85, "b": 60},
    )

    html = fig.to_html(
        include_plotlyjs="cdn",
        full_html=False,
        config={
            "scrollZoom": True,
            "displaylogo": False,
            "responsive": False,
        },
    )

    display(HTML(
        '<div style="width:100%; overflow-x:auto; border:1px solid #444;">'
        + html
        + '</div>'
    ))

    return fig


## Run analysis

In [8]:
result = analyze_song(AUDIO_PATH)
print_analysis_summary(result)


ValueError: No valid segmentation found. Increase SECTION_TOLERANCE_MS, reduce MINIMUM_SECTION_POINTS, or improve onset matching.

## Visualize all timing sections

In [ ]:
segment_figure = visualize_timing_segments(
    result,
    pixels_per_second=PIXELS_PER_SECOND,
)


### Optional: inspect only part of a long song

The full plot is scrollable. For focused inspection, pass a time range:


In [ ]:
# Example: inspect 45 to 75 seconds.
# segment_figure = visualize_timing_segments(
#     result,
#     time_range=(45, 75),
#     pixels_per_second=70,
# )


## Optional tolerance sweep

In [ ]:
def tolerance_sweep(
    result: dict,
    start_ms: int = 15,
    stop_ms: int = 1,
) -> list[dict]:
    rows: list[dict] = []
    matches = result["matches"]

    for tolerance_ms in range(start_ms, stop_ms - 1, -1):
        try:
            sections = minimum_timing_sections(
                beat_indices=matches["beat_indices"],
                beat_times=matches["peak_times"],
                tolerance_seconds=tolerance_ms / 1000,
                minimum_section_points=MINIMUM_SECTION_POINTS,
                error_quantile=ERROR_QUANTILE,
                integer_bpm=USE_INTEGER_BPM,
            )
            rows.append({
                "tolerance_ms": tolerance_ms,
                "section_count": len(sections),
                "status": "ok",
            })
            print(f"Tolerance {tolerance_ms:2d} ms: {len(sections)} section(s)")
        except ValueError:
            rows.append({
                "tolerance_ms": tolerance_ms,
                "section_count": None,
                "status": "no valid segmentation",
            })
            print(f"Tolerance {tolerance_ms:2d} ms: no valid segmentation")
            break

    return rows


# Uncomment when tuning a new song.
# sweep_results = tolerance_sweep(result)
